# Tratamento dos Dados — Casos por ano de diagóstico, faixa etária, escolaridade e raça cor


---
## 1. Imports

In [1]:
from __future__ import annotations

import re
import unicodedata
from io import StringIO
from pathlib import Path

import pandas as pd

---
## 2. Configurações Gerais


In [2]:
# Caminhos
PASTA_ENTRADA = Path("/home/anna/tcc/Limpeza e Integração/Casos")
PASTA_SAIDA = Path("/home/anna/tcc/Seleção e Transformação/Casos")

# Subpastas de cada dimensão
PASTA_ANO_DIAGNOSTICO = PASTA_ENTRADA / "Ano Diagnóstico" / "Dados"
PASTA_ESCOLARIDADE = PASTA_ENTRADA / "Escolaridade" / "Dados"
PASTA_FAIXA_ETARIA = PASTA_ENTRADA / "Faixa Etária" / "Dados"
PASTA_RACA_COR = PASTA_ENTRADA / "Raça-Cor" / "Dados"

# Arquivos de saída
SAIDA_ANO_DIAGNOSTICO = PASTA_SAIDA / "tabela_casos_por_ano_de_diagnóstico.csv"
SAIDA_ESCOLARIDADE = PASTA_SAIDA / "tabela_casos_por_escolaridade.csv"
SAIDA_FAIXA_ETARIA = PASTA_SAIDA / "tabela_casos_por_faixa_etária.csv"
SAIDA_RACA_COR = PASTA_SAIDA / "tabela_casos_por_raça_e_cor.csv"

# Faixas etárias padronizadas, destino do agrupamento
FAIXAS_PADRAO = [
    "0-9",
    "10-19",
    "20-29",
    "30-39",
    "40-49",
    "50-59",
    "60-69",
    "70 OU MAIS",
]

# Nomes padronizados das categorias (caixa alta e acentuação correta)
CATEGORIA_PADRAO: dict[str, str] = {
    "fundamental completo": "FUNDAMENTAL COMPLETO",
    "fundamental incompleto": "FUNDAMENTAL INCOMPLETO",
    "nenhuma": "NENHUMA",
    "nivel medio": "NÍVEL MÉDIO",
    "nivel superior completo": "NÍVEL SUPERIOR COMPLETO",
    "nivel superior incompleto": "NÍVEL SUPERIOR INCOMPLETO",
    "sem informacao": "SEM INFORMAÇÃO",
    "amarela": "AMARELA",
    "branca": "BRANCA",
    "indigena": "INDÍGENA",
    "parda": "PARDA",
    "preta": "PRETA",
}

# Intervalo válido de anos de diagnóstico
ANO_MIN_VALIDO = 1985
ANO_MAX_VALIDO = 2023

PASTA_SAIDA.mkdir(parents=True, exist_ok=True)


---
## 3. Funções de Normalização de Texto

Funções utilitárias para padronizar texto, nomes de colunas e extrair informações dos campos brutos.  
Todas as demais etapas dependem dessas funções — por isso ficam definidas primeiro.

In [3]:
def normalizar_texto(valor: str) -> str:
    """Remove acentos, converte para minúsculas e colapsa espaços múltiplos."""
    texto = str(valor).strip().lower()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("utf-8")
    texto = re.sub(r"\s+", " ", texto)
    return texto


def limpar_nome_coluna(coluna: str) -> str:
    """Remove BOM e aspas e normaliza o nome da coluna, sem alterar maiúsculas."""
    coluna = str(coluna).replace("﻿", "").replace('"', "").strip()
    coluna = unicodedata.normalize("NFKD", coluna).encode("ascii", "ignore").decode("utf-8")
    coluna = re.sub(r"\s+", " ", coluna).strip()
    return coluna


def extrair_municipio_e_codigo(procedencia: str) -> tuple[str | None, int | None]:
    """Extrai o município (normalizado) e o código IBGE de 'MG - 3100000 - Nome'; (None, None) se não casar."""
    texto = str(procedencia).replace('"', "").strip()
    correspondencia = re.match(r"^MG\s*-\s*(\d{7})\s*-\s*(.+)$", texto, flags=re.IGNORECASE)
    if not correspondencia:
        return None, None
    codigo = int(correspondencia.group(1))
    municipio = normalizar_texto(correspondencia.group(2))
    return municipio, codigo


def extrair_ano(nome_arquivo: str) -> int:
    """Extrai o ano (19xx ou 20xx) do nome do arquivo; levanta ValueError se não encontrar."""
    correspondencia = re.search(r"(19\d{2}|20\d{2})", nome_arquivo)
    if not correspondencia:
        raise ValueError(f"Ano não encontrado no nome do arquivo: {nome_arquivo}")
    return int(correspondencia.group(1))


---
## 4. Funções de Leitura dos Arquivos Brutos

Os arquivos brutos do RHC/INCA chegam como CSVs com encoding `latin1`, separador `;` e
linhas de cabeçalho não padronizadas. A função `ler_tabela_bruta` localiza automaticamente
a linha de cabeçalho real (que contém a coluna de procedência) e descarta as linhas anteriores.

In [4]:
def ler_tabela_bruta(caminho_arquivo: Path) -> pd.DataFrame:
    """Lê um arquivo bruto do RHC/INCA e acrescenta MUNICIPIOS e CODIGO a partir da procedência, descartando linhas sem código IBGE."""
    conteudo = caminho_arquivo.read_text(encoding="latin1", errors="ignore").splitlines()
    indice_cabecalho = None
    for indice, linha in enumerate(conteudo):
        if ";" in linha and "proced" in normalizar_texto(linha):
            indice_cabecalho = indice
            break
    if indice_cabecalho is None:
        raise ValueError(f"Cabeçalho não encontrado em: {caminho_arquivo}")

    texto_tabela = "\n".join(conteudo[indice_cabecalho:])
    df = pd.read_csv(StringIO(texto_tabela), sep=";", engine="python", encoding="latin1")
    df.columns = [limpar_nome_coluna(coluna) for coluna in df.columns]

    colunas_sem_nome = []
    for coluna in df.columns:
        if limpar_nome_coluna(coluna) == "" or re.match(r"(?i)^unnamed", coluna):
            colunas_sem_nome.append(coluna)
    if colunas_sem_nome:
        df = df.drop(columns=colunas_sem_nome)

    coluna_procedencia = df.columns[0]
    extraidos = df[coluna_procedencia].apply(extrair_municipio_e_codigo)
    df["MUNICIPIOS"] = extraidos.map(lambda par: par[0])
    df["CODIGO"] = extraidos.map(lambda par: par[1])

    df = df[df["CODIGO"].notna()].copy()
    df["CODIGO"] = df["CODIGO"].astype("Int64")
    df["MUNICIPIOS"] = df["MUNICIPIOS"].astype(str)

    return df


def colunas_numericas(df: pd.DataFrame, ignorar: set[str]) -> list[str]:
    """Devolve as colunas do df, exceto as de `ignorar` e a coluna 'total'."""
    colunas_do_arquivo = []
    for coluna in df.columns:
        if coluna in ignorar:
            continue
        if normalizar_texto(coluna) in {"total", ""}:
            continue
        colunas_do_arquivo.append(coluna)
    return colunas_do_arquivo


def listar_arquivos_ordenados(pasta: Path, padrao: str) -> list[Path]:
    """Lista os arquivos da pasta que casam com o padrão, ordenados pelo ano no nome."""
    arquivos = list(pasta.glob(padrao))
    if not arquivos:
        raise FileNotFoundError(f"Nenhum arquivo encontrado em: {pasta}/{padrao}")
    return sorted(arquivos, key=lambda caminho: extrair_ano(caminho.name))


---
## 5. Mapeamento de Faixas Etárias

Os arquivos brutos usam faixas de 5 em 5 anos (0-4, 5-9, 10-14 …). Esta função
agrupa essas faixas nas 8 categorias padronizadas definidas em `FAIXAS_PADRAO`,
unificando 70-79 e 80+ em **"70 ou mais"**.

In [5]:
def mapear_faixa(coluna: str) -> str | None:
    """Mapeia a faixa etária bruta (de 5 em 5 anos) para uma das 8 faixas padrão."""
    chave = normalizar_texto(coluna).replace(" ", "")
    mapa = {
        "00-04": "0-9", "0-4": "0-9", "05-09": "0-9", "5-9": "0-9",
        "10-14": "10-19", "15-19": "10-19",
        "20-24": "20-29", "25-29": "20-29",
        "30-34": "30-39", "35-39": "30-39",
        "40-44": "40-49", "45-49": "40-49",
        "50-54": "50-59", "55-59": "50-59",
        "60-64": "60-69", "65-69": "60-69",
        "70-74": "70 OU MAIS", "75-79": "70 OU MAIS", "70-79": "70 OU MAIS",
        "80-84": "70 OU MAIS", "85+": "70 OU MAIS", "85-89": "70 OU MAIS",
        "90+": "70 OU MAIS", "80oumais": "70 OU MAIS", "80+": "70 OU MAIS",
    }
    return mapa.get(chave)


---
## 6. Funções de Agregação

### 6.1 Ano de Diagnóstico
Agrega todos os arquivos anuais em uma única tabela com uma coluna por ano (1985–2023),
somando os casos por município.

In [ ]:
def renomear_coluna_ano_diagnostico(coluna: str) -> str:
    """Renomeia a coluna bruta para '<ano>_DIAGNOSTICO' ou 'ANO_SEM-INFORMAÇÃO'."""
    nome_limpo = limpar_nome_coluna(coluna)
    if re.fullmatch(r"\d{4}", nome_limpo):
        return f"{nome_limpo}_DIAGNOSTICO"
    categoria = normalizar_categoria(coluna)
    if categoria == "SEM INFORMAÇÃO":
        return "ANO_SEM-INFORMAÇÃO"
    return categoria


def agregar_ano_diagnostico(arquivos_brutos: list[Path]) -> pd.DataFrame:
    """Concatena os arquivos de Ano de Diagnóstico em uma tabela, descartando anos fora do período válido."""
    tabelas_por_arquivo: list[pd.DataFrame] = []
    for caminho_arquivo in arquivos_brutos:
        df = ler_tabela_bruta(caminho_arquivo)
        colunas_do_arquivo = colunas_numericas(df, {df.columns[0], "MUNICIPIOS", "CODIGO"})
        mapa_renomeacao = {}
        for coluna in colunas_do_arquivo:
            mapa_renomeacao[coluna] = renomear_coluna_ano_diagnostico(coluna)
        tabela_atual = df[["MUNICIPIOS", "CODIGO", *colunas_do_arquivo]].rename(columns=mapa_renomeacao)
        tabelas_por_arquivo.append(tabela_atual)

    # Cada arquivo bruto cobre um ano de referência do RHC, mas pode trazer
    # casos diagnosticados em outros anos (histórico); por isso empilha todos
    # os arquivos e soma por município, em vez de só concatenar colunas.
    tabela_empilhada = pd.concat(tabelas_por_arquivo, ignore_index=True)
    colunas_de_valor = []
    for coluna in tabela_empilhada.columns:
        if coluna not in {"MUNICIPIOS", "CODIGO"}:
            colunas_de_valor.append(coluna)
    for coluna in colunas_de_valor:
        tabela_empilhada[coluna] = pd.to_numeric(tabela_empilhada[coluna], errors="coerce")

    agregado = tabela_empilhada.groupby(["MUNICIPIOS", "CODIGO"], as_index=False)[colunas_de_valor].sum(min_count=1)

    # Anos válidos primeiro, depois as demais colunas; anos fora do período válido são descartados
    colunas_ano = []
    for coluna in colunas_de_valor:
        eh_coluna_de_ano = re.fullmatch(r"\d{4}_DIAGNOSTICO", str(coluna))
        if eh_coluna_de_ano and ANO_MIN_VALIDO <= int(str(coluna)[:4]) <= ANO_MAX_VALIDO:
            colunas_ano.append(coluna)
    colunas_ano = sorted(colunas_ano, key=lambda coluna: int(str(coluna)[:4]))

    # Colunas que não são "<ano>_DIAGNOSTICO" dentro do período (ex.:
    # ANO_SEM-INFORMAÇÃO) entram depois das colunas de ano, em ordem alfabética.
    colunas_restantes = []
    for coluna in colunas_de_valor:
        if coluna not in colunas_ano and not re.fullmatch(r"\d{4}_DIAGNOSTICO", str(coluna)):
            colunas_restantes.append(coluna)
    colunas_restantes = sorted(colunas_restantes)
    colunas_ordenadas = ["MUNICIPIOS", "CODIGO", *colunas_ano, *colunas_restantes]

    return agregado[colunas_ordenadas].rename(columns={"MUNICIPIOS": "MUNICÍPIO", "CODIGO": "CÓDIGO IBGE"})


### 6.2 Dimensões categóricas (Escolaridade, Raça/Cor e Faixa Etária)

Agrega os arquivos anuais em formato **wide**: cada coluna recebe o nome `YYYY_categoria`.
Para Faixa Etária, as colunas de 5 em 5 anos são agrupadas nas 8 faixas padronizadas.

In [ ]:
def agregar_largo_por_ano(
    arquivos_brutos: list[Path],
    mapeador_categoria=None,
    *,
    modo_faixa: bool = False,
) -> pd.DataFrame:
    """Monta a tabela larga (colunas 'AAAA_categoria') a partir dos arquivos brutos, um por ano; modo_faixa agrupa nas 8 faixas padrão."""
    resultado: pd.DataFrame | None = None

    for caminho_arquivo in sorted(arquivos_brutos, key=lambda caminho: extrair_ano(caminho.name)):
        ano = extrair_ano(caminho_arquivo.name)
        df = ler_tabela_bruta(caminho_arquivo)
        colunas_do_arquivo = colunas_numericas(df, {df.columns[0], "MUNICIPIOS", "CODIGO"})

        trabalho = df[["MUNICIPIOS", "CODIGO", *colunas_do_arquivo]].copy()
        for coluna in colunas_do_arquivo:
            trabalho[coluna] = pd.to_numeric(trabalho[coluna], errors="coerce")

        if modo_faixa:
            # Agrupa as colunas de 5 em 5 anos do arquivo bruto nas 8 faixas
            # etárias padrão, somando cada grupo por município; faixa sem
            # nenhuma coluna de origem naquele arquivo fica com NA (não 0).
            colunas_agrupadas: dict[str, list[str]] = {}
            for faixa in FAIXAS_PADRAO:
                colunas_agrupadas[faixa] = []
            for coluna in colunas_do_arquivo:
                faixa_mapeada = mapear_faixa(coluna)
                if faixa_mapeada is not None:
                    colunas_agrupadas[faixa_mapeada].append(coluna)

            tabela_atual = trabalho[["MUNICIPIOS", "CODIGO"]].copy()
            for faixa_destino in FAIXAS_PADRAO:
                colunas_origem = colunas_agrupadas[faixa_destino]
                if colunas_origem:
                    tabela_atual[f"{ano}_{faixa_destino}"] = trabalho[colunas_origem].sum(axis=1, min_count=1)
                else:
                    tabela_atual[f"{ano}_{faixa_destino}"] = pd.NA
        else:
            # Demais dimensões (Escolaridade, Raça/Cor): cada coluna bruta
            # vira "<ano>_<categoria normalizada>", sem agrupamento.
            colunas_renomeadas = {}
            for coluna in colunas_do_arquivo:
                colunas_renomeadas[coluna] = f"{ano}_{mapeador_categoria(coluna)}"
            tabela_atual = trabalho[["MUNICIPIOS", "CODIGO", *colunas_do_arquivo]].rename(columns=colunas_renomeadas)

        # Uma tabela por ano; junta todas num único DataFrame wide (outer join
        # para preservar municípios que só aparecem em alguns anos).
        if resultado is None:
            resultado = tabela_atual
        else:
            resultado = resultado.merge(tabela_atual, on=["MUNICIPIOS", "CODIGO"], how="outer")

    if resultado is None:
        raise ValueError("Nenhum arquivo processado.")

    return resultado.rename(columns={"MUNICIPIOS": "MUNICÍPIO", "CODIGO": "CÓDIGO IBGE"})


def normalizar_categoria(coluna: str) -> str:
    """Padroniza o nome da categoria em caixa alta e com acentuação correta (ver CATEGORIA_PADRAO)."""
    chave = normalizar_texto(limpar_nome_coluna(coluna))
    return CATEGORIA_PADRAO.get(chave, limpar_nome_coluna(coluna).upper())


---
## 6.3 Total por ano

Adiciona, ao final de cada tabela wide, uma coluna `TOTAL_YYYY` para cada ano presente
(seja o próprio ano, como em Ano de Diagnóstico, seja o conjunto `YYYY_categoria`,
como em Escolaridade/Faixa Etária/Raça-Cor). Cada `TOTAL_YYYY` é a soma das colunas
daquele ano **na própria linha** (ou seja, o total do município naquele ano, somando
as categorias) — não o total do estado inteiro.

In [8]:
def adicionar_colunas_total_ano(df: pd.DataFrame) -> pd.DataFrame:
    """Acrescenta a coluna 'AAAA_TOTAL' (soma das categorias do ano, por município) para cada ano."""
    df = df.copy()
    anos_encontrados = set()
    for coluna in df.columns:
        correspondencia = re.match(r"^(\d{4})(?:_|$)", str(coluna))
        if correspondencia:
            anos_encontrados.add(correspondencia.group(1))

    for ano in sorted(anos_encontrados):
        colunas_do_ano = []
        for coluna in df.columns:
            if re.match(rf"^{ano}(_|$)", str(coluna)):
                colunas_do_ano.append(coluna)
        df[f"{ano}_TOTAL"] = df[colunas_do_ano].sum(axis=1, min_count=1)
    return df


---
## 6.4 Regional de Saúde

Adiciona a coluna `REGIONAL DE SAÚDE` a cada tabela, via join pelo Código IBGE com
`tabela_municipios_regionais.csv` — mesma base de referência usada pelas outras
etapas do projeto para mapear município → Regional.

In [9]:
CAMINHO_MUNICIPIOS_REGIONAIS = Path("/home/anna/tcc/Seleção e Transformação/Municípios/tabela_municipios_regionais.csv")


def carregar_mapa_regional() -> pd.Series:
    """Carrega o mapa CÓDIGO IBGE → REGIONAL DE SAÚDE a partir da tabela de municípios."""
    mapa = pd.read_csv(CAMINHO_MUNICIPIOS_REGIONAIS, encoding="utf-8-sig")
    mapa["CÓDIGO IBGE"] = pd.to_numeric(mapa["CÓDIGO IBGE"], errors="coerce").astype("Int64")
    return mapa.set_index("CÓDIGO IBGE")["REGIONAL NORMALIZADA"].rename("REGIONAL DE SAÚDE")


def adicionar_coluna_regional(df: pd.DataFrame) -> pd.DataFrame:
    """Adiciona REGIONAL DE SAÚDE logo após CÓDIGO IBGE, via join pelo código IBGE."""
    mapa_regional = carregar_mapa_regional()
    df = df.copy()
    df["CÓDIGO IBGE"] = pd.to_numeric(df["CÓDIGO IBGE"], errors="coerce").astype("Int64")
    df["REGIONAL DE SAÚDE"] = df["CÓDIGO IBGE"].map(mapa_regional)

    sem_regional = int(df["REGIONAL DE SAÚDE"].isna().sum())
    if sem_regional > 0:
        print(f"   ⚠️ {sem_regional} município(s) sem Regional de Saúde correspondente.")

    colunas = df.columns.tolist()
    colunas.remove("REGIONAL DE SAÚDE")
    posicao = colunas.index("CÓDIGO IBGE") + 1
    colunas.insert(posicao, "REGIONAL DE SAÚDE")
    return df[colunas]

---
## 7. Pipeline de Tratamento

Executa o pipeline completo em 5 etapas:
1. Localiza os arquivos brutos de cada dimensão
2. Agrega Ano de Diagnóstico
3. Agrega Escolaridade, Faixa Etária e Raça/Cor (formato wide)
4. Adiciona `YYYY_TOTAL` ao final de cada tabela
5. Adiciona `REGIONAL DE SAÚDE` e salva os CSVs de saída (um arquivo por dimensão)

In [10]:
# Etapa 1: localizar arquivos
print("Arquivos brutos:")
arquivos_ano_diagnostico = listar_arquivos_ordenados(PASTA_ANO_DIAGNOSTICO, "AnoDiagnostico-*.csv")
arquivos_escolaridade = listar_arquivos_ordenados(PASTA_ESCOLARIDADE, "Escolaridade-*.csv")
arquivos_faixa_etaria = listar_arquivos_ordenados(PASTA_FAIXA_ETARIA, "Faixa-Etaria-*.csv")
arquivos_raca_cor = listar_arquivos_ordenados(PASTA_RACA_COR, "RacaCor-*.csv")

print(f"   Ano Diagnóstico : {len(arquivos_ano_diagnostico)} arquivo(s)")
print(f"   Escolaridade    : {len(arquivos_escolaridade)} arquivo(s)")
print(f"   Faixa Etária    : {len(arquivos_faixa_etaria)} arquivo(s)")
print(f"   Raça/Cor        : {len(arquivos_raca_cor)} arquivo(s)")


Arquivos brutos:
   Ano Diagnóstico : 38 arquivo(s)
   Escolaridade    : 38 arquivo(s)
   Faixa Etária    : 38 arquivo(s)
   Raça/Cor        : 38 arquivo(s)


In [11]:
# Etapa 2: Ano de Diagnóstico
print("\nProcessando Ano de Diagnóstico...")
df_ano_diagnostico = agregar_ano_diagnostico(arquivos_ano_diagnostico)
print(f"   Shape: {df_ano_diagnostico.shape} | Municípios: {df_ano_diagnostico['MUNICÍPIO'].nunique()}")
df_ano_diagnostico.head(3)


Processando Ano de Diagnóstico...


   Shape: (853, 42) | Municípios: 853


,MUNICÍPIO,CÓDIGO IBGE,1985_DIAGNOSTICO,1986_DIAGNOSTICO,1987_DIAGNOSTICO,1988_DIAGNOSTICO,1989_DIAGNOSTICO,1990_DIAGNOSTICO,1991_DIAGNOSTICO,1992_DIAGNOSTICO,...,2015_DIAGNOSTICO,2016_DIAGNOSTICO,2017_DIAGNOSTICO,2018_DIAGNOSTICO,2019_DIAGNOSTICO,2020_DIAGNOSTICO,2021_DIAGNOSTICO,2022_DIAGNOSTICO,2023_DIAGNOSTICO,ANO_SEM-INFORMAÇÃO
0,abadia dos dourados,3100104,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,2.0,2.0,1.0,0.0,1.0,0.0,2.0,0.0
1,abaete,3100203,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,6.0,7.0,3.0,2.0,3.0,5.0,6.0,14.0,9.0,4.0
2,abre campo,3100302,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,6.0,4.0,4.0,3.0,5.0,4.0,3.0,3.0,1.0


In [12]:
# Etapa 3a: Escolaridade
print("Processando Escolaridade...")
df_escolaridade = agregar_largo_por_ano(arquivos_escolaridade, normalizar_categoria)
print(f"   Shape: {df_escolaridade.shape} | Municípios: {df_escolaridade['MUNICÍPIO'].nunique()}")

Processando Escolaridade...


   Shape: (853, 163) | Municípios: 853


In [13]:
# Etapa 3b: Faixa Etária
print("Processando Faixa Etária...")
df_faixa_etaria = agregar_largo_por_ano(arquivos_faixa_etaria, modo_faixa=True)
print(f"   Shape: {df_faixa_etaria.shape} | Municípios: {df_faixa_etaria['MUNICÍPIO'].nunique()}")

Processando Faixa Etária...


   Shape: (853, 306) | Municípios: 853


In [14]:
# Etapa 3c: Raça/Cor
print("Processando Raça/Cor...")
df_raca_cor = agregar_largo_por_ano(arquivos_raca_cor, normalizar_categoria)
print(f"   Shape: {df_raca_cor.shape} | Municípios: {df_raca_cor['MUNICÍPIO'].nunique()}")

Processando Raça/Cor...


   Shape: (853, 150) | Municípios: 853


In [15]:
# Etapa 4: Total por ano
print("\nAdicionando YYYY_TOTAL por dimensão...")
df_ano_diagnostico = adicionar_colunas_total_ano(df_ano_diagnostico)
df_escolaridade = adicionar_colunas_total_ano(df_escolaridade)
df_faixa_etaria = adicionar_colunas_total_ano(df_faixa_etaria)
df_raca_cor = adicionar_colunas_total_ano(df_raca_cor)


Adicionando YYYY_TOTAL por dimensão...


In [16]:
# Etapa 4b: Regional de Saúde
print("\nAdicionando REGIONAL DE SAÚDE por dimensão...")
df_ano_diagnostico = adicionar_coluna_regional(df_ano_diagnostico)
df_escolaridade = adicionar_coluna_regional(df_escolaridade)
df_faixa_etaria = adicionar_coluna_regional(df_faixa_etaria)
df_raca_cor = adicionar_coluna_regional(df_raca_cor)


Adicionando REGIONAL DE SAÚDE por dimensão...


In [17]:
# Etapa 5: Salvar CSVs
print("\n💾 Salvando arquivos...")

df_ano_diagnostico.to_csv(SAIDA_ANO_DIAGNOSTICO, index=False)
df_escolaridade.to_csv(SAIDA_ESCOLARIDADE, index=False)
df_faixa_etaria.to_csv(SAIDA_FAIXA_ETARIA, index=False)
df_raca_cor.to_csv(SAIDA_RACA_COR, index=False)


💾 Salvando arquivos...


---
## 8. Inspeção Pós-Processamento

Verificação rápida dos arquivos gerados: shape, primeiras linhas e contagem de valores ausentes.

In [18]:
from IPython.display import display, Markdown

tabelas = {
    "Ano de Diagnóstico": df_ano_diagnostico,
    "Escolaridade": df_escolaridade,
    "Faixa Etária": df_faixa_etaria,
    "Raça/Cor": df_raca_cor,
}

for nome, df in tabelas.items():
    nulos = int(df.isnull().sum().sum())
    display(Markdown(f"**{nome}** — {df.shape[0]} linhas × {df.shape[1]} colunas | nulos totais: {nulos}"))
    display(df.head(2))

**Ano de Diagnóstico** — 853 linhas × 82 colunas | nulos totais: 598

,MUNICÍPIO,CÓDIGO IBGE,REGIONAL DE SAÚDE,1985_DIAGNOSTICO,1986_DIAGNOSTICO,1987_DIAGNOSTICO,1988_DIAGNOSTICO,1989_DIAGNOSTICO,1990_DIAGNOSTICO,1991_DIAGNOSTICO,...,2014_TOTAL,2015_TOTAL,2016_TOTAL,2017_TOTAL,2018_TOTAL,2019_TOTAL,2020_TOTAL,2021_TOTAL,2022_TOTAL,2023_TOTAL
0,abadia dos dourados,3100104,uberlandia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,2.0,2.0,1.0,0.0,1.0,0.0,2.0
1,abaete,3100203,sete lagoas,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,6.0,6.0,7.0,3.0,2.0,3.0,5.0,6.0,14.0,9.0


**Escolaridade** — 853 linhas × 190 colunas | nulos totais: 54922

,MUNICÍPIO,CÓDIGO IBGE,REGIONAL DE SAÚDE,1998_SEM INFORMAÇÃO,1999_NÍVEL MÉDIO,1999_SEM INFORMAÇÃO,2000_FUNDAMENTAL COMPLETO,2000_FUNDAMENTAL INCOMPLETO,2000_NENHUMA,2000_NÍVEL MÉDIO,...,2014_TOTAL,2015_TOTAL,2016_TOTAL,2017_TOTAL,2018_TOTAL,2019_TOTAL,2020_TOTAL,2021_TOTAL,2022_TOTAL,2023_TOTAL
0,abadia dos dourados,3100104,uberlandia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,2.0,1.0,1.0,2.0,1.0,NaN,NaN,2.0,2.0
1,abaete,3100203,sete lagoas,NaN,NaN,NaN,0.0,0.0,0.0,0.0,...,7.0,4.0,7.0,2.0,2.0,2.0,7.0,8.0,14.0,14.0


**Faixa Etária** — 853 linhas × 345 colunas | nulos totais: 175863

,MUNICÍPIO,CÓDIGO IBGE,REGIONAL DE SAÚDE,1985_0-9,1985_10-19,1985_20-29,1985_30-39,1985_40-49,1985_50-59,1985_60-69,...,2014_TOTAL,2015_TOTAL,2016_TOTAL,2017_TOTAL,2018_TOTAL,2019_TOTAL,2020_TOTAL,2021_TOTAL,2022_TOTAL,2023_TOTAL
0,abadia dos dourados,3100104,uberlandia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,None,2.0,1.0,1.0,2.0,1.0,NaN,NaN,2.0,2.0
1,abaete,3100203,sete lagoas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,7.0,4.0,7.0,2.0,2.0,2.0,7.0,8.0,14.0,14.0


**Raça/Cor** — 853 linhas × 177 colunas | nulos totais: 53949

,MUNICÍPIO,CÓDIGO IBGE,REGIONAL DE SAÚDE,1998_BRANCA,1998_PARDA,1998_PRETA,1998_SEM INFORMAÇÃO,1999_BRANCA,1999_PRETA,1999_SEM INFORMAÇÃO,...,2014_TOTAL,2015_TOTAL,2016_TOTAL,2017_TOTAL,2018_TOTAL,2019_TOTAL,2020_TOTAL,2021_TOTAL,2022_TOTAL,2023_TOTAL
0,abadia dos dourados,3100104,uberlandia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,2.0,1.0,1.0,2.0,1.0,NaN,NaN,2.0,2.0
1,abaete,3100203,sete lagoas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,7.0,4.0,7.0,2.0,2.0,2.0,7.0,8.0,14.0,14.0
